In [8]:
from datetime import datetime
import json
import os
import traceback
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# -----------------------------------------------------------------------------
# Configuration & Hyperparameters
# -----------------------------------------------------------------------------
TEST_SIZE = 0.9  # Change test size as needed (e.g., 0.2, 0.3)
RANDOM_STATE = 69
EVAL_METRIC = "mlogloss"
OBJECTIVE = "multi:softprob"

# You can define a single dict OR a list of dicts here:
HYPERPARAMETERS_TO_TEST = [{
        "learning_rate": 0.5,
        "max_depth": 6,
        "num_class": 6,
        "n_estimators": 300,
        "subsample": 0.6,
        "colsample_bytree": 0.6,
        "min_child_weight": 6,
        "reg_lambda": 1.0,
        "reg_alpha": 0,
        "gamma": 2,
    },
    {
        "learning_rate": 0.1,
        "max_depth": 5,
        "num_class": 6,
        "n_estimators": 75,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "reg_lambda": 1.0,
        "reg_alpha": 0.1,
        "gamma": 2,
    },]

# File Paths
RAW_DATA_PATH = r"../dataset/full.csv"
PRUNED_DATA_PATH = r"../dataset/full_v5.csv"
PCA_DATA_PATH = r"../dataset/full_v5_pca.csv"
OUTPUT_RESULTS_PATH = r"../dataset/xgb_gss_results.json"


# -----------------------------------------------------------------------------
# Utility Functions
# -----------------------------------------------------------------------------
def compute_metrics(y_true, y_pred, class_names):
    """Calculates accuracy, precision, recall, macro F1, and normalized confusion matrix dict."""
    acc = float(accuracy_score(y_true, y_pred))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_pct = np.true_divide(cm, cm.sum(axis=1, keepdims=True)) * 100
        cm_pct = np.nan_to_num(cm_pct)

    cm_dict = {
        str(class_names[i]): {
            str(class_names[j]): float(round(cm_pct[i, j], 2))
            for j in range(len(class_names))
        }
        for i in range(len(class_names))
    }

    return {
        "accuracy": round(acc, 4),
        "precision_macro": round(float(precision), 4),
        "recall_macro": round(float(recall), 4),
        "f1_macro": round(float(f1), 4),
        "confusion_matrix_pct": cm_dict,
    }


def save_progress_atomically(file_path: str, instances: list):
    """Saves output atomically via temporary file."""
    temp_path = f"{file_path}.tmp"
    output_payload = {
        "experiment_count": len(instances),
        "instances": instances,
    }
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(output_payload, f, indent=4)
    os.replace(temp_path, file_path)


# Ensure HYPERPARAMETERS_TO_TEST is always a list
if isinstance(HYPERPARAMETERS_TO_TEST, dict):
    HYPERPARAMETERS_TO_TEST = [HYPERPARAMETERS_TO_TEST]

# -----------------------------------------------------------------------------
# Data Loading & Preprocessing
# -----------------------------------------------------------------------------
raw_data = pd.read_csv(RAW_DATA_PATH, index_col=False)
pruned_data = pd.read_csv(PRUNED_DATA_PATH, index_col=False).drop(columns=["split"], errors="ignore")
pca_data = pd.read_csv(PCA_DATA_PATH, index_col=False).drop(columns=["split"], errors="ignore")

groups = raw_data["subject"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(raw_data["Activity"])

X_raw = raw_data.drop(columns=["Activity", "subject"], errors="ignore").select_dtypes(include=[np.number])
X_pruned = pruned_data.drop(columns=["Activity", "subject"], errors="ignore").select_dtypes(include=[np.number])
X_pca = pca_data.drop(columns=["Activity", "subject"], errors="ignore").select_dtypes(include=[np.number])

datasets = {
    "raw_data_results": X_raw,
    "pruned_data_results": X_pruned,
    "pca_data_results": X_pca,
}

# -----------------------------------------------------------------------------
# GroupShuffleSplit Setup
# -----------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_raw, y_encoded, groups=groups))

split_info = {
    "test_size_ratio": TEST_SIZE,
    "train_size_ratio": round(1.0 - TEST_SIZE, 2),
    "random_state": RANDOM_STATE,
    "n_train_samples": int(len(train_idx)),
    "n_test_samples": int(len(test_idx)),
    "n_train_groups": int(len(np.unique(groups.iloc[train_idx]))),
    "n_test_groups": int(len(np.unique(groups.iloc[test_idx]))),
}

# -----------------------------------------------------------------------------
# Resumption & Execution Logic
# -----------------------------------------------------------------------------
instances = []
completed_experiment_keys = set()

if os.path.exists(OUTPUT_RESULTS_PATH):
    try:
        with open(OUTPUT_RESULTS_PATH, "r", encoding="utf-8") as f:
            existing_payload = json.load(f)
            instances = existing_payload.get("instances", [])
            for inst in instances:
                # Deduplicate based on hyperparams AND split settings combined
                exp_key = json.dumps({
                    "params": inst.get("parameters"),
                    "split": inst.get("split_config")
                }, sort_keys=True)
                completed_experiment_keys.add(exp_key)
        print(f"Loaded {len(instances)} existing experiments from {OUTPUT_RESULTS_PATH}.")
    except Exception as e:
        print(f"Could not read existing file ({e}). Starting fresh.")
        instances = []

# Main Loop
for idx, param in enumerate(HYPERPARAMETERS_TO_TEST):
    exp_key = json.dumps({"params": param, "split": split_info}, sort_keys=True)
    
    if exp_key in completed_experiment_keys:
        print(f"Skipping set #{idx + 1}: Exact same parameters and split settings already executed.")
        continue

    print(f"\n=== Running Experiment Set #{idx + 1}/{len(HYPERPARAMETERS_TO_TEST)} ===")
    instance = {
        "instance_id": len(instances) + 1,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "split_config": split_info,
        "parameters": param,
    }

    param_clean = {
        k: v for k, v in param.items()
        if k not in ["eval_metric", "objective", "random_state", "num_rounds", "num_boost_round"]
    }

    for ds_name, X_df in datasets.items():
        print(f"  -> Testing dataset variant: {ds_name}")
        try:
            X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
            y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

            model = xgb.XGBClassifier(
                **param_clean,
                random_state=RANDOM_STATE,
                eval_metric=EVAL_METRIC,
                objective=OBJECTIVE,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            metrics = compute_metrics(y_test, y_pred, label_encoder.classes_)

            instance[ds_name] = {"status": "success", "metrics": metrics}

        except Exception as e:
            print(f"  [ERROR] {ds_name} failed: {e}")
            traceback.print_exc()
            instance[ds_name] = {"status": "failed", "error": str(e), "metrics": None}

    instances.append(instance)
    completed_experiment_keys.add(exp_key)
    save_progress_atomically(OUTPUT_RESULTS_PATH, instances)
    print(f"--> Successfully appended Instance #{instance['instance_id']} to {OUTPUT_RESULTS_PATH}")

print("\nAll experiments complete.")

Loaded 12 existing experiments from ../dataset/xgb_gss_results.json.

=== Running Experiment Set #1/2 ===
  -> Testing dataset variant: raw_data_results
  -> Testing dataset variant: pruned_data_results
  -> Testing dataset variant: pca_data_results
--> Successfully appended Instance #13 to ../dataset/xgb_gss_results.json

=== Running Experiment Set #2/2 ===
  -> Testing dataset variant: raw_data_results
  -> Testing dataset variant: pruned_data_results
  -> Testing dataset variant: pca_data_results
--> Successfully appended Instance #14 to ../dataset/xgb_gss_results.json

All experiments complete.
